# Laboratorio 4: Simulación de eventos discretos
## 5.4 Simulación de eventos discretos en SimPy

**Curso:** Simulación Computacional · Universidad de los Llanos · **Estudiante:** Duván Felipe Baquero · **Código:** 160005002

Contenido del cuaderno, en el orden de la guía:

1. Instalación de SimPy.
2. Tutorial *Event discrete simulation with SimPy* de Stefan Scherfke (video y diapositivas de EuroPython 2014),
   desarrollado paso a paso junto con el cuaderno *Discrete-event simulation with SimPy.ipynb* del curso.
3. Tutorial *SimPy in 10 minutes* de la documentación oficial.
4. Los siete ejemplos pedidos de *All Examples – SimPy*: Bank Renege, Carwash, Machine Shop, Movie Renege,
   Gas Station Refueling, Process Communication y Event Latency.

En cada ejemplo el código es el de la documentación oficial de SimPy 4 (se conserva tal cual para que la
salida se pueda comparar con la publicada); antes de cada uno se explica qué modela y qué elementos de SimPy
usa, y después se comenta la salida obtenida.

---
### 1. Instalación

En Google Colab basta con `pip install simpy`. En este equipo se instaló igual, con el `pip` de Python 3.13.

In [1]:
%pip install -q simpy
import simpy, sys, random
print("Python", sys.version.split()[0], "| SimPy", simpy.__version__)

Note: you may need to restart the kernel to use updated packages.
Python 3.13.7 | SimPy 4.1.2



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: C:\Users\duvan\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


---
### 2. Tutorial de Stefan Scherfke: *Discrete-event simulation with SimPy*

SimPy es un *framework* de simulación de eventos discretos basado en procesos. Sus piezas principales son:

- **Environment**: el reloj de la simulación y la cola de eventos. `env.now` es el tiempo simulado.
- **Process**: un proceso se describe con una **función generadora** de Python. Cada vez que el proceso hace
  `yield` de un evento, queda suspendido hasta que ese evento ocurra.
- **Event**: algo que ocurre en un instante; el más común es `env.timeout(d)`, que ocurre $d$ unidades de tiempo
  después.
- **Resource**: recursos compartidos por los que los procesos hacen cola (Resource, Container y Store).

#### 2.1 Funciones generadoras

El `yield` entrega un valor y congela la función hasta la siguiente llamada. Con `send()` se puede meter un
valor de vuelta. Es el mecanismo con el que SimPy suspende y reanuda los procesos.

In [2]:
def generator(x):
    y = yield x + 1      # entrega x+1 y espera un valor
    return y + 1

g = generator(1)
print("next(g) ->", next(g))
try:
    g.send(3)
except StopIteration as fin:
    print("g.send(3) -> StopIteration con valor", fin.value)

next(g) -> 2
g.send(3) -> StopIteration con valor 4


In [3]:
# ejemplo del cuaderno del curso: un generador que duplica lo que recibe
def double_inputs():
    while True:
        x = yield
        yield x * 2

gen = double_inputs()
next(gen)                     # correr hasta la primera cesión
print(gen.send(5))            # entra en 'x'
next(gen)
print(gen.send(6))
next(gen)
print(gen.send(94.3))

10
12
188.6


#### 2.2 Primer modelo: dos relojes

Cada reloj es un proceso que imprime la hora y se duerme `tick` unidades de tiempo. El entorno intercala los dos
procesos según el tiempo simulado, no según el orden del código.

In [4]:
def clock(env, name, tick):
    while True:
        print(name, env.now)
        yield env.timeout(tick)

env = simpy.Environment()
env.process(clock(env, "rápido", 0.5))
env.process(clock(env, "lento", 1))
env.run(until=2)

rápido 0
lento 0
rápido 0.5
lento 1
rápido 1.0
rápido 1.5


#### 2.3 `Environment` y `RealtimeEnvironment`

`Environment` simula "tan rápido como sea posible". `RealtimeEnvironment` sincroniza el tiempo simulado con el
reloj de pared; con `factor=0.1` cada unidad simulada dura 0,1 segundos reales. Se mide el tiempo real de las
dos versiones con el mismo modelo:

In [5]:
import time

def tic(env):
    while True:
        yield env.timeout(1)

for nombre, entorno in [("Environment", simpy.Environment()),
                        ("RealtimeEnvironment(factor=0.1)", simpy.rt.RealtimeEnvironment(factor=0.1, strict=False))]:
    entorno.process(tic(entorno))
    inicio = time.perf_counter()
    entorno.run(until=10)
    print(f"{nombre:34s}: 10 unidades simuladas en {time.perf_counter() - inicio:.3f} s reales")

Environment                       : 10 unidades simuladas en 0.000 s reales


RealtimeEnvironment(factor=0.1)   : 10 unidades simuladas en 1.000 s reales


#### 2.4 Los eventos *timeout* dejan pasar el tiempo

Un orador espera hasta la hora de inicio de su charla y luego habla 30 minutos.

In [6]:
def speaker(env, start):
    print("t:", env.now, "- el orador espera su turno")
    until_start = start - env.now
    yield env.timeout(until_start)
    print("t:", env.now, "- empieza la charla")
    yield env.timeout(30)
    print("t:", env.now, "- termina la charla")

env = simpy.Environment()
env.process(speaker(env, 5))
env.run(100)

t: 0 - el orador espera su turno
t: 5 - empieza la charla
t: 35 - termina la charla


#### 2.5 Los procesos también son eventos

Un proceso puede esperar a que otro termine haciendo `yield` del proceso, y recibe su valor de retorno. Aquí el
moderador espera a cada orador y recibe el material (*handout*) que este entrega al final.

In [7]:
def speaker(env):
    yield env.timeout(30)
    return "handout"

def moderator(env):
    for i in range(3):
        val = yield env.process(speaker(env))
        print(f"t = {env.now}: el moderador recibió '{val}' del orador {i}")

env = simpy.Environment()
env.process(moderator(env))
env.run()

t = 30: el moderador recibió 'handout' del orador 0
t = 60: el moderador recibió 'handout' del orador 1
t = 90: el moderador recibió 'handout' del orador 2


#### 2.6 Interrupciones asíncronas

Un proceso puede interrumpir a otro con `interrupt()`. El interrumpido recibe una excepción `simpy.Interrupt`,
que puede atrapar. Aquí cada charla dura entre 25 y 30 minutos (aleatorio), y el moderador corta a todos los
oradores a los 30.

In [8]:
random.seed(7)

def speaker(env):
    try:
        yield env.timeout(random.randint(25, 35))
        print(f"t = {env.now}: el orador terminó a tiempo")
    except simpy.Interrupt as interrupt:
        print(f"t = {env.now}: {interrupt.cause}")

def moderator(env):
    for i in range(3):
        speaker_proc = env.process(speaker(env))
        yield env.timeout(30)
        speaker_proc.interrupt("¡No queda tiempo!")

env = simpy.Environment()
env.process(moderator(env))
try:
    env.run()
except RuntimeError as error:
    print(f"t = {env.now}: ERROR -> {error}")

t = 30: ¡No queda tiempo!
t = 57: el orador terminó a tiempo
t = 60: ERROR -> <Process(speaker) object at 0x29a3f612120> has terminated and cannot be interrupted.


Aquí aparece el problema que motiva el siguiente paso. El primer orador fue cortado en $t = 30$, pero el
segundo terminó por su cuenta en $t = 57$, antes de sus 30 minutos. En $t = 60$ el moderador igual intenta
interrumpirlo y SimPy lanza un `RuntimeError`, porque no se puede interrumpir un proceso que ya terminó. Lo
resuelven los **eventos de condición**.

#### 2.7 Eventos de condición

`evento_a | evento_b` es un evento que ocurre cuando ocurre cualquiera de los dos (`&` exige los dos). El
moderador espera "que el orador termine **o** se cumplan 30 minutos", y solo interrumpe si el orador no está en
los resultados.

In [9]:
random.seed(3)

def speaker(env):
    try:
        duracion = random.randint(25, 35)
        yield env.timeout(duracion)
        print(f"t = {env.now}: el orador terminó a tiempo ({duracion} min)")
    except simpy.Interrupt as interrupt:
        print(f"t = {env.now}: {interrupt.cause}")

def moderator(env):
    for i in range(5):
        speaker_proc = env.process(speaker(env))
        results = yield speaker_proc | env.timeout(30)
        if speaker_proc not in results:
            speaker_proc.interrupt("¡No queda tiempo! El moderador corta la charla")

env = simpy.Environment()
env.process(moderator(env))
try:
    env.run()
except RuntimeError as error:
    print(f"t = {env.now}: ERROR -> {error}")

t = 28: el orador terminó a tiempo (28 min)
t = 58: ¡No queda tiempo! El moderador corta la charla
t = 88: ¡No queda tiempo! El moderador corta la charla
t = 115: el orador terminó a tiempo (27 min)
t = 145: el orador terminó a tiempo (30 min)
t = 145: ERROR -> <Process(speaker) object at 0x29a3f61ccb0> has terminated and cannot be interrupted.


Funciona para los cuatro primeros oradores, pero el quinto habló **exactamente** 30 minutos y el error volvió a
aparecer en $t = 145$. En ese instante hay dos eventos: el fin de la charla y el *timeout* del moderador. El
evento de condición se dispara con el primero que SimPy procesa (el *timeout*, que se creó antes), así que
`results` no incluye al orador; pero antes de que el moderador reanude, SimPy procesa también el fin de la
charla. Cuando el moderador llama a `interrupt()`, el orador ya terminó. Es un caso de **empate** que el código
de las diapositivas no contempla. La solución es preguntar si el proceso sigue vivo (`is_alive`), que es
justamente lo que imprime el cuaderno del curso antes de interrumpir:

In [10]:
random.seed(3)

def moderator(env):
    for i in range(5):
        speaker_proc = env.process(speaker(env))
        results = yield speaker_proc | env.timeout(30)
        if speaker_proc not in results and speaker_proc.is_alive:
            speaker_proc.interrupt("¡No queda tiempo! El moderador corta la charla")

env = simpy.Environment()
env.process(moderator(env))
env.run()

t = 28: el orador terminó a tiempo (28 min)
t = 58: ¡No queda tiempo! El moderador corta la charla
t = 88: ¡No queda tiempo! El moderador corta la charla
t = 115: el orador terminó a tiempo (27 min)
t = 145: el orador terminó a tiempo (30 min)


#### 2.8 Recursos compartidos

SimPy trae tres tipos de recursos. En todos, los procesos que no pueden ser atendidos hacen cola:

- **Resource**: un número limitado de "puestos" (cajeros, máquinas). Se pide con `request()` y se libera al
  salir del bloque `with`.
- **Container**: una cantidad homogénea (litros, unidades) de la que se saca con `get(n)` y se añade con
  `put(n)`.
- **Store**: objetos individuales que se guardan con `put(obj)` y se sacan con `get()`.

Un ejemplo corto de cada uno:

In [11]:
def usuario(env, nombre, recurso, duracion):
    with recurso.request() as req:
        yield req
        print(f"t = {env.now}: {nombre} toma el recurso")
        yield env.timeout(duracion)
        print(f"t = {env.now}: {nombre} lo libera")

env = simpy.Environment()
cajero = simpy.Resource(env, capacity=1)
for k in range(3):
    env.process(usuario(env, f"cliente {k}", cajero, 4))
env.run()
print("-" * 40)

def consumidor(env, tanque):
    for _ in range(3):
        yield tanque.get(40)
        print(f"t = {env.now}: saqué 40 L, quedan {tanque.level} L")
        yield env.timeout(1)

def proveedor(env, tanque):
    yield env.timeout(5)
    yield tanque.put(60)
    print(f"t = {env.now}: el proveedor llenó 60 L, hay {tanque.level} L")

env = simpy.Environment()
tanque = simpy.Container(env, capacity=100, init=70)
env.process(consumidor(env, tanque)); env.process(proveedor(env, tanque))
env.run()
print("-" * 40)

def productor(env, bodega):
    for k in range(3):
        yield env.timeout(2)
        yield bodega.put(f"pieza {k}")
        print(f"t = {env.now}: se guardó la pieza {k}")

def cliente(env, bodega):
    while True:
        pieza = yield bodega.get()
        print(f"t = {env.now}: el cliente se llevó '{pieza}'")

env = simpy.Environment()
bodega = simpy.Store(env)
env.process(productor(env, bodega)); env.process(cliente(env, bodega))
env.run()

t = 0: cliente 0 toma el recurso
t = 4: cliente 0 lo libera
t = 4: cliente 1 toma el recurso
t = 8: cliente 1 lo libera
t = 8: cliente 2 toma el recurso
t = 12: cliente 2 lo libera
----------------------------------------
t = 0: saqué 40 L, quedan 30 L
t = 5: el proveedor llenó 60 L, hay 50 L
t = 5: saqué 40 L, quedan 50 L
t = 6: saqué 40 L, quedan 10 L
----------------------------------------
t = 2: se guardó la pieza 0
t = 2: el cliente se llevó 'pieza 0'
t = 4: se guardó la pieza 1
t = 4: el cliente se llevó 'pieza 1'
t = 6: se guardó la pieza 2
t = 6: el cliente se llevó 'pieza 2'


#### 2.9 Ejemplo: asistente a la conferencia

Cada asistente ve tres charlas seguidas. En cada una gana algo de conocimiento (menos si tiene hambre) y le da
más hambre. Luego va al *buffet*, come y vuelve a empezar.

In [12]:
from random import randint
random.seed(2014)

TALKS_PER_SESSION = 3
TALK_LENGTH = 30
BREAK_LENGTH = 15

def attendee(env, name, knowledge=0, hunger=0):
    while True:
        # asistir a las charlas
        for i in range(TALKS_PER_SESSION):
            knowledge += randint(0, 3) / (1 + hunger)
            hunger += randint(1, 4)
            yield env.timeout(TALK_LENGTH)
        print(f"t = {env.now:3d}: el asistente {name} terminó las charlas con conocimiento {knowledge:.2f} y hambre {hunger:.2f}")
        # ir al buffet
        food = randint(3, 12)
        hunger -= min(food, hunger)
        yield env.timeout(BREAK_LENGTH)
        print(f"t = {env.now:3d}: el asistente {name} terminó de comer con hambre {hunger:.2f}")

env = simpy.Environment()
for i in range(5):
    env.process(attendee(env, i))
env.run(until=220)

t =  90: el asistente 0 terminó las charlas con conocimiento 2.54 y hambre 8.00
t =  90: el asistente 1 terminó las charlas con conocimiento 2.08 y hambre 9.00
t =  90: el asistente 2 terminó las charlas con conocimiento 2.03 y hambre 10.00
t =  90: el asistente 3 terminó las charlas con conocimiento 3.73 y hambre 6.00
t =  90: el asistente 4 terminó las charlas con conocimiento 0.14 y hambre 9.00
t = 105: el asistente 0 terminó de comer con hambre 5.00
t = 105: el asistente 1 terminó de comer con hambre 0.00
t = 105: el asistente 2 terminó de comer con hambre 1.00
t = 105: el asistente 3 terminó de comer con hambre 0.00
t = 105: el asistente 4 terminó de comer con hambre 2.00
t = 195: el asistente 0 terminó las charlas con conocimiento 2.69 y hambre 8.00
t = 195: el asistente 1 terminó las charlas con conocimiento 2.82 y hambre 6.00
t = 195: el asistente 2 terminó las charlas con conocimiento 3.40 y hambre 8.00
t = 195: el asistente 3 terminó las charlas con conocimiento 4.73 y hambre

#### 2.10 Ejemplo: asistente a la conferencia en el *buffet*

Ahora el *buffet* es un `Resource` con un solo puesto y comer tarda 3 minutos. Cada asistente espera el
puesto **o** a que se le acabe el descanso (evento de condición); si no alcanza a comer, le da más hambre.

En las diapositivas la línea `time_left = LEN_BREAK - (env.now - start)` usa una constante `LEN_BREAK` que no
está definida; debe ser `BREAK_LENGTH`, como ya aparece corregido en el cuaderno del curso.

In [13]:
random.seed(2014)
DURATION_EAT = 3
BUFFET_SLOTS = 1

def attendee(env, name, buffet, knowledge=0, hunger=0):
    while True:
        for i in range(TALKS_PER_SESSION):
            knowledge += randint(0, 3) / (1 + hunger)
            hunger += randint(1, 4)
            yield env.timeout(TALK_LENGTH)
        print(f"t = {env.now:3d}: el asistente {name} terminó las charlas con conocimiento {knowledge:.2f} y hambre {hunger:.2f}")

        start = env.now
        with buffet.request() as req:
            yield req | env.timeout(BREAK_LENGTH - DURATION_EAT)
            time_left = BREAK_LENGTH - (env.now - start)
            if req.triggered:
                food = min(randint(3, 12), time_left)      # menos tiempo -> menos comida
                yield env.timeout(DURATION_EAT)
                hunger -= min(food, hunger)
                time_left -= DURATION_EAT
                print(f"t = {env.now:3d}: el asistente {name} terminó de comer con hambre {hunger:.2f}")
            else:
                hunger += 1      # castigo por solo mirar la comida
                print(f"t = {env.now:3d}: el asistente {name} no alcanzó a comer, su hambre sube a {hunger:.2f}")
        yield env.timeout(time_left)

env = simpy.Environment()
buffet = simpy.Resource(env, capacity=BUFFET_SLOTS)
for i in range(5):
    env.process(attendee(env, i, buffet))
env.run(until=220)

t =  90: el asistente 0 terminó las charlas con conocimiento 2.54 y hambre 8.00
t =  90: el asistente 1 terminó las charlas con conocimiento 2.08 y hambre 9.00
t =  90: el asistente 2 terminó las charlas con conocimiento 2.03 y hambre 10.00
t =  90: el asistente 3 terminó las charlas con conocimiento 3.73 y hambre 6.00
t =  90: el asistente 4 terminó las charlas con conocimiento 0.14 y hambre 9.00
t =  93: el asistente 0 terminó de comer con hambre 5.00
t =  96: el asistente 1 terminó de comer con hambre 0.00
t =  99: el asistente 2 terminó de comer con hambre 1.00
t = 102: el asistente 3 terminó de comer con hambre 0.00
t = 102: el asistente 4 no alcanzó a comer, su hambre sube a 10.00
t = 195: el asistente 0 terminó las charlas con conocimiento 3.02 y hambre 9.00
t = 195: el asistente 1 terminó las charlas con conocimiento 2.08 y hambre 6.00
t = 195: el asistente 2 terminó las charlas con conocimiento 2.92 y hambre 8.00
t = 195: el asistente 3 terminó las charlas con conocimiento 4.8


Al comparar con el ejemplo anterior se ve el efecto del recurso. Sin restricción, los cinco asistentes comían
al mismo tiempo y todos terminaban en $t = 105$. Con un solo puesto comen uno detrás de otro, en $t = 93, 96, 99$
y $102$. Al quinto se le acaba la ventana de $15 - 3 = 12$ minutos justo en $t = 102$ y se queda sin comer; su
hambre sube a 10 y, en la segunda sesión, a 17. Además siempre es el mismo (el asistente 4) porque todos
terminan las charlas en el mismo instante y SimPy atiende las peticiones en el orden en que se hicieron. Con 3
minutos por persona y 12 de ventana, el *buffet* solo alcanza para 4 asistentes por descanso. Harían falta dos
puestos o un descanso más largo.


---
### 3. Tutorial *SimPy in 10 minutes*

#### 3.1 Conceptos básicos: un carro que estaciona y conduce

El proceso `car` alterna entre estacionar 5 unidades de tiempo y conducir 2.

In [14]:
def car(env):
    while True:
        print(f"Empieza a estacionar en {env.now}")
        parking_duration = 5
        yield env.timeout(parking_duration)

        print(f"Empieza a conducir en {env.now}")
        trip_duration = 2
        yield env.timeout(trip_duration)

env = simpy.Environment()
env.process(car(env))
env.run(until=15)

Empieza a estacionar en 0
Empieza a conducir en 5
Empieza a estacionar en 7
Empieza a conducir en 12
Empieza a estacionar en 14


#### 3.2 Interacción entre procesos: esperar a otro proceso e interrumpirlo

Ahora el carro es eléctrico: mientras está estacionado carga la batería, y la carga es un proceso aparte
(`charge`) al que el carro espera. Un conductor impaciente interrumpe la carga en $t = 3$.

In [15]:
class Car:
    def __init__(self, env):
        self.env = env
        # al crear el carro se arranca su proceso run()
        self.action = env.process(self.run())

    def run(self):
        while True:
            print(f"Empieza a estacionar y cargar en {self.env.now}")
            charge_duration = 5
            try:
                yield self.env.process(self.charge(charge_duration))
            except simpy.Interrupt:
                print("Carga interrumpida. Ojalá la batería tenga suficiente...")

            print(f"Empieza a conducir en {self.env.now}")
            trip_duration = 2
            yield self.env.timeout(trip_duration)

    def charge(self, duration):
        yield self.env.timeout(duration)

def driver(env, car):
    yield env.timeout(3)
    car.action.interrupt()

env = simpy.Environment()
car = Car(env)
env.process(driver(env, car))
env.run(until=15)

Empieza a estacionar y cargar en 0
Carga interrumpida. Ojalá la batería tenga suficiente...
Empieza a conducir en 3
Empieza a estacionar y cargar en 5
Empieza a conducir en 10
Empieza a estacionar y cargar en 12


#### 3.3 Recursos compartidos: estación de carga con dos puestos

Cuatro carros llegan a una estación de carga (BCS) que solo tiene dos puestos. Llegan en $t = 0, 2, 4, 6$ y
cada carga dura 5.

In [16]:
def car(env, name, bcs, driving_time, charge_duration):
    # conducir hasta la estación
    yield env.timeout(driving_time)

    # pedir uno de los puestos de carga
    print(f"{name} llega en {env.now}")
    with bcs.request() as req:
        yield req
        print(f"{name} empieza a cargar en {env.now}")
        yield env.timeout(charge_duration)
        print(f"{name} sale de la estación en {env.now}")

env = simpy.Environment()
bcs = simpy.Resource(env, capacity=2)
for i in range(4):
    env.process(car(env, f"Carro {i}", bcs, i * 2, 5))
env.run()

Carro 0 llega en 0
Carro 0 empieza a cargar en 0
Carro 1 llega en 2
Carro 1 empieza a cargar en 2
Carro 2 llega en 4
Carro 0 sale de la estación en 5
Carro 2 empieza a cargar en 5
Carro 3 llega en 6
Carro 1 sale de la estación en 7
Carro 3 empieza a cargar en 7
Carro 2 sale de la estación en 10
Carro 3 sale de la estación en 12


El carro 2 llega en $t = 4$ con los dos puestos ocupados y espera hasta $t = 5$, cuando sale el carro 0. El
carro 3 llega en $t = 6$ y espera hasta $t = 7$, cuando sale el carro 1. Cada uno espera exactamente una unidad
de tiempo, que es lo que se obtiene haciendo la cuenta a mano.

---
### 4. Ejemplos de SimPy

#### 4.1 Bank Renege (clientes que abandonan la cola)

Un banco con un solo mostrador (`Resource` de capacidad 1). Llegan 5 clientes con tiempos entre llegadas
exponenciales de media 10. Cada cliente tiene una paciencia aleatoria entre 1 y 3: espera el mostrador **o** a
que se le acabe la paciencia (evento de condición `req | timeout`). Si se le acaba primero, abandona
(*renege*). El servicio es exponencial de media 12.

Elementos de SimPy: `Resource`, eventos de condición.

In [17]:
# Código del ejemplo oficial: https://simpy.readthedocs.io/en/latest/examples/bank_renege.html
import random

import simpy

RANDOM_SEED = 42
NEW_CUSTOMERS = 5  # Total number of customers
INTERVAL_CUSTOMERS = 10.0  # Generate new customers roughly every x seconds
MIN_PATIENCE = 1  # Min. customer patience
MAX_PATIENCE = 3  # Max. customer patience


def source(env, number, interval, counter):
    """Source generates customers randomly"""
    for i in range(number):
        c = customer(env, f'Customer{i:02d}', counter, time_in_bank=12.0)
        env.process(c)
        t = random.expovariate(1.0 / interval)
        yield env.timeout(t)


def customer(env, name, counter, time_in_bank):
    """Customer arrives, is served and leaves."""
    arrive = env.now
    print(f'{arrive:7.4f} {name}: Here I am')

    with counter.request() as req:
        patience = random.uniform(MIN_PATIENCE, MAX_PATIENCE)
        # Wait for the counter or abort at the end of our tether
        results = yield req | env.timeout(patience)

        wait = env.now - arrive

        if req in results:
            # We got to the counter
            print(f'{env.now:7.4f} {name}: Waited {wait:6.3f}')

            tib = random.expovariate(1.0 / time_in_bank)
            yield env.timeout(tib)
            print(f'{env.now:7.4f} {name}: Finished')

        else:
            # We reneged
            print(f'{env.now:7.4f} {name}: RENEGED after {wait:6.3f}')


# Setup and start the simulation
print('Bank renege')
random.seed(RANDOM_SEED)
env = simpy.Environment()

# Start processes and run
counter = simpy.Resource(env, capacity=1)
env.process(source(env, NEW_CUSTOMERS, INTERVAL_CUSTOMERS, counter))
env.run()

Bank renege
 0.0000 Customer00: Here I am
 0.0000 Customer00: Waited  0.000
 3.8595 Customer00: Finished
10.2006 Customer01: Here I am
10.2006 Customer01: Waited  0.000
12.7265 Customer02: Here I am
13.9003 Customer02: RENEGED after  1.174
23.7507 Customer01: Finished
34.9993 Customer03: Here I am
34.9993 Customer03: Waited  0.000
37.9599 Customer03: Finished
40.4798 Customer04: Here I am
40.4798 Customer04: Waited  0.000
43.1401 Customer04: Finished



**Salida.** De los 5 clientes, 4 fueron atendidos sin esperar y uno abandonó. El cliente 2 llegó en
$t = 12{,}73$ mientras el cliente 1 ocupaba el mostrador (estuvo desde 10,20 hasta 23,75). Su paciencia era de
1,17 y se fue en $t = 13{,}90$. El evento de condición `req | env.timeout(patience)` es el que decide:
`req in results` indica si se obtuvo el mostrador antes de agotar la paciencia. Con servicios de media 12 y
llegadas cada 10 en promedio ($\rho = 1{,}2$), en una corrida más larga los abandonos serían frecuentes.


#### 4.2 Carwash (lavadero de carros)

Un lavadero con 2 máquinas. Al comienzo hay 4 carros y luego llega uno cada 5 a 9 minutos. Lavar un carro
toma 5 minutos. La clase `Carwash` encapsula el recurso y el proceso de lavado; cada carro pide una máquina y
espera a que el proceso `wash` termine. Se simulan 20 minutos.

Elementos de SimPy: `Resource`, esperar a otro proceso (`yield env.process(...)`).

In [18]:
# Código del ejemplo oficial: https://simpy.readthedocs.io/en/latest/examples/carwash.html
import itertools
import random

import simpy

# fmt: off
RANDOM_SEED = 42
NUM_MACHINES = 2  # Number of machines in the carwash
WASHTIME = 5      # Minutes it takes to clean a car
T_INTER = 7       # Create a car every ~7 minutes
SIM_TIME = 20     # Simulation time in minutes
# fmt: on


class Carwash:
    """A carwash has a limited number of machines (``NUM_MACHINES``) to
    clean cars in parallel.

    Cars have to request one of the machines. When they got one, they
    can start the washing processes and wait for it to finish (which
    takes ``washtime`` minutes).

    """

    def __init__(self, env, num_machines, washtime):
        self.env = env
        self.machine = simpy.Resource(env, num_machines)
        self.washtime = washtime

    def wash(self, car):
        """The washing processes. It takes a ``car`` processes and tries
        to clean it."""
        yield self.env.timeout(self.washtime)
        pct_dirt = random.randint(50, 99)
        print(f"Carwash removed {pct_dirt}% of {car}'s dirt.")


def car(env, name, cw):
    """The car process (each car has a ``name``) arrives at the carwash
    (``cw``) and requests a cleaning machine.

    It then starts the washing process, waits for it to finish and
    leaves to never come back ...

    """
    print(f'{name} arrives at the carwash at {env.now:.2f}.')
    with cw.machine.request() as request:
        yield request

        print(f'{name} enters the carwash at {env.now:.2f}.')
        yield env.process(cw.wash(name))

        print(f'{name} leaves the carwash at {env.now:.2f}.')


def setup(env, num_machines, washtime, t_inter):
    """Create a carwash, a number of initial cars and keep creating cars
    approx. every ``t_inter`` minutes."""
    # Create the carwash
    carwash = Carwash(env, num_machines, washtime)

    car_count = itertools.count()

    # Create 4 initial cars
    for _ in range(4):
        env.process(car(env, f'Car {next(car_count)}', carwash))

    # Create more cars while the simulation is running
    while True:
        yield env.timeout(random.randint(t_inter - 2, t_inter + 2))
        env.process(car(env, f'Car {next(car_count)}', carwash))


# Setup and start the simulation
print('Carwash')
print('Check out http://youtu.be/fXXmeP9TvBg while simulating ... ;-)')
random.seed(RANDOM_SEED)  # This helps to reproduce the results

# Create an environment and start the setup process
env = simpy.Environment()
env.process(setup(env, NUM_MACHINES, WASHTIME, T_INTER))

# Execute!
env.run(until=SIM_TIME)

Carwash
Check out http://youtu.be/fXXmeP9TvBg while simulating ... ;-)
Car 0 arrives at the carwash at 0.00.
Car 1 arrives at the carwash at 0.00.
Car 2 arrives at the carwash at 0.00.
Car 3 arrives at the carwash at 0.00.
Car 0 enters the carwash at 0.00.
Car 1 enters the carwash at 0.00.
Car 4 arrives at the carwash at 5.00.
Carwash removed 97% of Car 0's dirt.
Carwash removed 67% of Car 1's dirt.
Car 0 leaves the carwash at 5.00.
Car 1 leaves the carwash at 5.00.
Car 2 enters the carwash at 5.00.
Car 3 enters the carwash at 5.00.
Car 5 arrives at the carwash at 10.00.
Carwash removed 64% of Car 2's dirt.
Carwash removed 58% of Car 3's dirt.
Car 2 leaves the carwash at 10.00.
Car 3 leaves the carwash at 10.00.
Car 4 enters the carwash at 10.00.
Car 5 enters the carwash at 10.00.
Carwash removed 97% of Car 4's dirt.
Carwash removed 56% of Car 5's dirt.
Car 4 leaves the carwash at 15.00.
Car 5 leaves the carwash at 15.00.
Car 6 arrives at the carwash at 16.00.
Car 6 enters the carwash 


**Salida.** Los 4 carros iniciales llegan en $t = 0$; los carros 0 y 1 entran de una vez y los 2 y 3 esperan
hasta $t = 5$. El carro 4 llega en $t = 5$ y espera hasta $t = 10$; desde ahí la cola desaparece, porque dos
máquinas atienden hasta 2 carros cada 5 minutos (0,4 por minuto) y llegan en promedio 1/7 = 0,14 por minuto. El
atraso se debe solo a los 4 carros iniciales. Aquí se ve bien que el carro espera al proceso de lavado con
`yield env.process(cw.wash(name))`: el lavado es un proceso distinto del carro.


#### 4.3 Machine Shop (taller con máquinas que se dañan)

Un taller con 10 máquinas iguales que fabrican piezas (tiempo normal de media 10 minutos). Cada máquina se
daña en promedio cada 300 minutos. Un único técnico las repara (30 minutos) y, cuando no hay nada dañado, hace
otros trabajos menos importantes. Las reparaciones **desplazan** esos trabajos: el técnico es un
`PreemptiveResource` y las reparaciones piden con prioridad 1 y los otros trabajos con prioridad 2. Se simulan
4 semanas.

Elementos de SimPy: interrupciones, `PreemptiveResource`.

In [19]:
# Código del ejemplo oficial: https://simpy.readthedocs.io/en/latest/examples/machine_shop.html
import random

import simpy

# fmt: off
RANDOM_SEED = 42
PT_MEAN = 10.0         # Avg. processing time in minutes
PT_SIGMA = 2.0         # Sigma of processing time
MTTF = 300.0           # Mean time to failure in minutes
BREAK_MEAN = 1 / MTTF  # Param. for expovariate distribution
REPAIR_TIME = 30.0     # Time it takes to repair a machine in minutes
JOB_DURATION = 30.0    # Duration of other jobs in minutes
NUM_MACHINES = 10      # Number of machines in the machine shop
WEEKS = 4              # Simulation time in weeks
SIM_TIME = WEEKS * 7 * 24 * 60  # Simulation time in minutes
# fmt: on


def time_per_part():
    """Return actual processing time for a concrete part."""
    t = random.normalvariate(PT_MEAN, PT_SIGMA)
    # The normalvariate can be negative, but we want only positive times.
    while t <= 0:
        t = random.normalvariate(PT_MEAN, PT_SIGMA)
    return t


def time_to_failure():
    """Return time until next failure for a machine."""
    return random.expovariate(BREAK_MEAN)


class Machine:
    """A machine produces parts and may get broken every now and then.

    If it breaks, it requests a *repairman* and continues the production
    after the it is repaired.

    A machine has a *name* and a number of *parts_made* thus far.

    """

    def __init__(self, env, name, repairman):
        self.env = env
        self.name = name
        self.parts_made = 0
        self.broken = False

        # Start "working" and "break_machine" processes for this machine.
        self.process = env.process(self.working(repairman))
        env.process(self.break_machine())

    def working(self, repairman):
        """Produce parts as long as the simulation runs.

        While making a part, the machine may break multiple times.
        Request a repairman when this happens.

        """
        while True:
            # Start making a new part
            done_in = time_per_part()
            while done_in:
                start = self.env.now
                try:
                    # Working on the part
                    yield self.env.timeout(done_in)
                    done_in = 0  # Set to 0 to exit while loop.

                except simpy.Interrupt:
                    self.broken = True
                    done_in -= self.env.now - start  # How much time left?

                    # Request a repairman. This will preempt its "other_job".
                    with repairman.request(priority=1) as req:
                        yield req
                        yield self.env.timeout(REPAIR_TIME)

                    self.broken = False

            # Part is done.
            self.parts_made += 1

    def break_machine(self):
        """Break the machine every now and then."""
        while True:
            yield self.env.timeout(time_to_failure())
            if not self.broken:
                # Only break the machine if it is currently working.
                self.process.interrupt()


def other_jobs(env, repairman):
    """The repairman's other (unimportant) job."""
    while True:
        # Start a new job
        done_in = JOB_DURATION
        while done_in:
            # Retry the job until it is done.
            # Its priority is lower than that of machine repairs.
            with repairman.request(priority=2) as req:
                yield req
                start = env.now
                try:
                    yield env.timeout(done_in)
                    done_in = 0
                except simpy.Interrupt:
                    done_in -= env.now - start


# Setup and start the simulation
print('Machine shop')
random.seed(RANDOM_SEED)  # This helps to reproduce the results

# Create an environment and start the setup process
env = simpy.Environment()
repairman = simpy.PreemptiveResource(env, capacity=1)
machines = [Machine(env, f'Machine {i}', repairman) for i in range(NUM_MACHINES)]
env.process(other_jobs(env, repairman))

# Execute!
env.run(until=SIM_TIME)

# Analysis/results
print(f'Machine shop results after {WEEKS} weeks')
for machine in machines:
    print(f'{machine.name} made {machine.parts_made} parts.')

Machine shop
Machine shop results after 4 weeks
Machine 0 made 3251 parts.
Machine 1 made 3273 parts.
Machine 2 made 3242 parts.
Machine 3 made 3343 parts.
Machine 4 made 3387 parts.
Machine 5 made 3244 parts.
Machine 6 made 3269 parts.
Machine 7 made 3185 parts.
Machine 8 made 3302 parts.
Machine 9 made 3279 parts.



**Salida.** En cuatro semanas (40.320 minutos) cada máquina hizo entre 3185 y 3387 piezas, 3277,5 en promedio.
Sin daños, a 10 minutos por pieza, cada máquina haría unas 4032, así que se pierde cerca del 19 % de la
producción. La razón está en el técnico: 10 máquinas que se dañan cada 300 minutos y tardan 30 en repararse le
generan $10 \cdot 30/300 = 1$ hora de reparación por cada hora de trabajo. Está prácticamente saturado, y las
máquinas no solo pierden los 30 minutos de reparación, sino también el tiempo de espera hasta que el técnico
las atiende. Sus "otros trabajos" quedan siempre en segundo plano: cada reparación (prioridad 1) los
interrumpe y se retoman después, donde quedaron. El ejemplo muestra dos mecanismos: `interrupt()` para dañar una máquina mientras trabaja, y
`PreemptiveResource` con prioridades.


#### 4.4 Movie Renege (taquilla de cine)

Una taquilla vende boletas para tres películas, 50 por película. Llegan espectadores con tiempos exponenciales
de media 0,5 minutos; cada uno quiere de 1 a 6 boletas para una película al azar. Cuando a una película le
quedan menos de 2 boletas se dispara un evento compartido de "agotada", y todos los que estaban haciendo fila
para esa película se van al tiempo. Se simulan 120 minutos.

Elementos de SimPy: `Resource`, eventos de condición, eventos compartidos entre procesos.

In [20]:
# Código del ejemplo oficial: https://simpy.readthedocs.io/en/latest/examples/movie_renege.html
import random
from typing import Dict, List, NamedTuple, Optional

import simpy

RANDOM_SEED = 42
TICKETS = 50  # Number of tickets per movie
SELLOUT_THRESHOLD = 2  # Fewer tickets than this is a sellout
SIM_TIME = 120  # Simulate until


def moviegoer(env, movie, num_tickets, theater):
    """A moviegoer tries to by a number of tickets (*num_tickets*) for
    a certain *movie* in a *theater*.

    If the movie becomes sold out, she leaves the theater. If she gets
    to the counter, she tries to buy a number of tickets. If not enough
    tickets are left, she argues with the teller and leaves.

    If at most one ticket is left after the moviegoer bought her
    tickets, the *sold out* event for this movie is triggered causing
    all remaining moviegoers to leave.

    """
    with theater.counter.request() as my_turn:
        # Wait until it's our turn or until the movie is sold out
        result = yield my_turn | theater.sold_out[movie]

        # Check if it's our turn or if movie is sold out
        if my_turn not in result:
            theater.num_renegers[movie] += 1
            return

        # Check if enough tickets left.
        if theater.available[movie] < num_tickets:
            # Moviegoer leaves after some discussion
            yield env.timeout(0.5)
            return

        # Buy tickets
        theater.available[movie] -= num_tickets
        if theater.available[movie] < SELLOUT_THRESHOLD:
            # Trigger the "sold out" event for the movie
            theater.sold_out[movie].succeed()
            theater.when_sold_out[movie] = env.now
            theater.available[movie] = 0
        yield env.timeout(1)


def customer_arrivals(env, theater):
    """Create new *moviegoers* until the sim time reaches 120."""
    while True:
        yield env.timeout(random.expovariate(1 / 0.5))

        movie = random.choice(theater.movies)
        num_tickets = random.randint(1, 6)
        if theater.available[movie]:
            env.process(moviegoer(env, movie, num_tickets, theater))


class Theater(NamedTuple):
    counter: simpy.Resource
    movies: List[str]
    available: Dict[str, int]
    sold_out: Dict[str, simpy.Event]
    when_sold_out: Dict[str, Optional[float]]
    num_renegers: Dict[str, int]


# Setup and start the simulation
print('Movie renege')
random.seed(RANDOM_SEED)
env = simpy.Environment()

# Create movie theater
movies = ['Python Unchained', 'Kill Process', 'Pulp Implementation']
theater = Theater(
    counter=simpy.Resource(env, capacity=1),
    movies=movies,
    available=dict.fromkeys(movies, TICKETS),
    sold_out={movie: env.event() for movie in movies},
    when_sold_out=dict.fromkeys(movies),
    num_renegers=dict.fromkeys(movies, 0),
)

# Start process and run
env.process(customer_arrivals(env, theater))
env.run(until=SIM_TIME)

# Analysis/results
for movie in movies:
    if theater.sold_out[movie]:
        sellout_time = theater.when_sold_out[movie]
        num_renegers = theater.num_renegers[movie]
        print(
            f'Movie "{movie}" sold out {sellout_time:.1f} minutes '
            f'after ticket counter opening.'
        )
        print(f'  Number of people leaving queue when film sold out: {num_renegers}')

Movie renege
Movie "Python Unchained" sold out 38.0 minutes after ticket counter opening.
  Number of people leaving queue when film sold out: 16
Movie "Kill Process" sold out 43.0 minutes after ticket counter opening.
  Number of people leaving queue when film sold out: 5
Movie "Pulp Implementation" sold out 28.0 minutes after ticket counter opening.
  Number of people leaving queue when film sold out: 5



**Salida.** Las tres películas se agotaron antes de los 120 minutos: *Pulp Implementation* a los 28 minutos,
*Python Unchained* a los 38 y *Kill Process* a los 43. Cuando se agotaron, se fueron de la fila 5, 16 y 5
personas, respectivamente. Las 16 personas de *Python Unchained* muestran por qué el evento compartido es útil:
un único evento `sold_out[movie]` avisa al mismo tiempo a todos los que esperaban esa película, que lo tienen
en su condición `yield my_turn | theater.sold_out[movie]`. Sin él, cada persona tendría que llegar a la taquilla
para enterarse.


#### 4.5 Gas Station Refueling (estación de gasolina)

Una estación con 2 surtidores que comparten un tanque de 200 litros (`Container`). Los carros llegan cada 30 a
300 segundos, piden un surtidor y cargan lo que les falta para llenar su tanque de 50 L a 2 L/s. Un proceso de
control revisa el nivel cada 10 segundos y, si baja del 25 %, llama a un carrotanque que tarda 300 segundos
en llegar y rellena el tanque.

Elementos de SimPy: `Resource`, `Container`, esperar a otro proceso.

In [21]:
# Código del ejemplo oficial: https://simpy.readthedocs.io/en/latest/examples/gas_station_refuel.html
import itertools
import random

import simpy

# fmt: off
RANDOM_SEED = 42
STATION_TANK_SIZE = 200    # Size of the gas station tank (liters)
THRESHOLD = 25             # Station tank minimum level (% of full)
CAR_TANK_SIZE = 50         # Size of car fuel tanks (liters)
CAR_TANK_LEVEL = [5, 25]   # Min/max levels of car fuel tanks (liters)
REFUELING_SPEED = 2        # Rate of refuelling car fuel tank (liters / second)
TANK_TRUCK_TIME = 300      # Time it takes tank truck to arrive (seconds)
T_INTER = [30, 300]        # Interval between car arrivals [min, max] (seconds)
SIM_TIME = 1000            # Simulation time (seconds)
# fmt: on


def car(name, env, gas_station, station_tank):
    """A car arrives at the gas station for refueling.

    It requests one of the gas station's fuel pumps and tries to get the
    desired amount of fuel from it. If the station's fuel tank is
    depleted, the car has to wait for the tank truck to arrive.

    """
    car_tank_level = random.randint(*CAR_TANK_LEVEL)
    print(f'{env.now:6.1f} s: {name} arrived at gas station')
    with gas_station.request() as req:
        # Request one of the gas pumps
        yield req

        # Get the required amount of fuel
        fuel_required = CAR_TANK_SIZE - car_tank_level
        yield station_tank.get(fuel_required)

        # The "actual" refueling process takes some time
        yield env.timeout(fuel_required / REFUELING_SPEED)

        print(f'{env.now:6.1f} s: {name} refueled with {fuel_required:.1f}L')


def gas_station_control(env, station_tank):
    """Periodically check the level of the gas station tank and call the tank
    truck if the level falls below a threshold."""
    while True:
        if station_tank.level / station_tank.capacity * 100 < THRESHOLD:
            # We need to call the tank truck now!
            print(f'{env.now:6.1f} s: Calling tank truck')
            # Wait for the tank truck to arrive and refuel the station tank
            yield env.process(tank_truck(env, station_tank))

        yield env.timeout(10)  # Check every 10 seconds


def tank_truck(env, station_tank):
    """Arrives at the gas station after a certain delay and refuels it."""
    yield env.timeout(TANK_TRUCK_TIME)
    amount = station_tank.capacity - station_tank.level
    station_tank.put(amount)
    print(
        f'{env.now:6.1f} s: Tank truck arrived and refuelled station with {amount:.1f}L'
    )


def car_generator(env, gas_station, station_tank):
    """Generate new cars that arrive at the gas station."""
    for i in itertools.count():
        yield env.timeout(random.randint(*T_INTER))
        env.process(car(f'Car {i}', env, gas_station, station_tank))


# Setup and start the simulation
print('Gas Station refuelling')
random.seed(RANDOM_SEED)

# Create environment and start processes
env = simpy.Environment()
gas_station = simpy.Resource(env, 2)
station_tank = simpy.Container(env, STATION_TANK_SIZE, init=STATION_TANK_SIZE)
env.process(gas_station_control(env, station_tank))
env.process(car_generator(env, gas_station, station_tank))

# Execute!
env.run(until=SIM_TIME)

Gas Station refuelling
  87.0 s: Car 0 arrived at gas station
 105.5 s: Car 0 refueled with 37.0L
 129.0 s: Car 1 arrived at gas station
 148.0 s: Car 1 refueled with 38.0L
 284.0 s: Car 2 arrived at gas station
 305.0 s: Car 2 refueled with 42.0L
 385.0 s: Car 3 arrived at gas station
 398.5 s: Car 3 refueled with 27.0L
 459.0 s: Car 4 arrived at gas station
 460.0 s: Calling tank truck
 481.0 s: Car 4 refueled with 44.0L
 705.0 s: Car 5 arrived at gas station
 750.0 s: Car 6 arrived at gas station
 760.0 s: Tank truck arrived and refuelled station with 188.0L
 779.0 s: Car 6 refueled with 38.0L
 781.5 s: Car 5 refueled with 43.0L
 891.0 s: Car 7 arrived at gas station
 904.0 s: Car 7 refueled with 26.0L



**Salida.** Los primeros cuatro carros cargaron 37, 38, 42 y 27 litros, y el tanque de 200 L quedó en 56. El
carro 4 llegó en $t = 459$ y sacó 44 L de una vez (el `get` del `Container` retira toda la cantidad pedida). El
tanque bajó a 12 L (6 %), y en la revisión de $t = 460$ el control llamó al carrotanque. Este llegó 300 s después,
en $t = 760$, y cargó 188 L. Mientras tanto llegaron los carros 5 y 6 (en 705 y 750), que no pudieron cargar
porque no había combustible suficiente: el `get` los dejó esperando en el `Container`. Terminaron en 781,5 y 779.
El carro 5 esperó 55 s por el combustible. El ejemplo muestra cómo un `Container` bloquea a los procesos hasta que
haya cantidad suficiente, sin programar esa lógica a mano.


#### 4.6 Process Communication (comunicación entre procesos)

Muestra cómo conectar procesos por medio de "tuberías" hechas con `Store`. Primero, un generador envía
mensajes a un consumidor (uno a uno). Después, con la clase `BroadcastPipe`, un mismo mensaje se copia a varios
consumidores (uno a muchos). Si el consumidor llega tarde a leer, el mensaje lo espera en el `Store`, y el
consumidor lo nota al comparar la hora de envío con la de recepción.

Elementos de SimPy: `Store`.

In [22]:
# Código del ejemplo oficial: https://simpy.readthedocs.io/en/latest/examples/process_communication.html
import random

import simpy
import simpy.core

RANDOM_SEED = 42
SIM_TIME = 100


class BroadcastPipe:
    """A Broadcast pipe that allows one process to send messages to many.

    This construct is useful when message consumers are running at
    different rates than message generators and provides an event
    buffering to the consuming processes.

    The parameters are used to create a new
    :class:`~simpy.resources.store.Store` instance each time
    :meth:`get_output_conn()` is called.

    """

    def __init__(self, env, capacity=simpy.core.Infinity):
        self.env = env
        self.capacity = capacity
        self.pipes = []

    def put(self, value):
        """Broadcast a *value* to all receivers."""
        if not self.pipes:
            raise RuntimeError('There are no output pipes.')
        events = [store.put(value) for store in self.pipes]
        return self.env.all_of(events)  # Condition event for all "events"

    def get_output_conn(self):
        """Get a new output connection for this broadcast pipe.

        The return value is a :class:`~simpy.resources.store.Store`.

        """
        pipe = simpy.Store(self.env, capacity=self.capacity)
        self.pipes.append(pipe)
        return pipe


def message_generator(name, env, out_pipe):
    """A process which randomly generates messages."""
    while True:
        # wait for next transmission
        yield env.timeout(random.randint(6, 10))

        # messages are time stamped to later check if the consumer was
        # late getting them.  Note, using event.triggered to do this may
        # result in failure due to FIFO nature of simulation yields.
        # (i.e. if at the same env.now, message_generator puts a message
        # in the pipe first and then message_consumer gets from pipe,
        # the event.triggered will be True in the other order it will be
        # False
        msg = (env.now, f'{name} says hello at {env.now}')
        out_pipe.put(msg)


def message_consumer(name, env, in_pipe):
    """A process which consumes messages."""
    while True:
        # Get event for message pipe
        msg = yield in_pipe.get()

        if msg[0] < env.now:
            # if message was already put into pipe, then
            # message_consumer was late getting to it. Depending on what
            # is being modeled this, may, or may not have some
            # significance
            print(
                f'LATE Getting Message: at time {env.now}: '
                f'{name} received message: {msg[1]}'
            )

        else:
            # message_consumer is synchronized with message_generator
            print(f'at time {env.now}: {name} received message: {msg[1]}.')

        # Process does some other work, which may result in missing messages
        yield env.timeout(random.randint(4, 8))


# Setup and start the simulation
print('Process communication')
random.seed(RANDOM_SEED)
env = simpy.Environment()

# For one-to-one or many-to-one type pipes, use Store
pipe = simpy.Store(env)
env.process(message_generator('Generator A', env, pipe))
env.process(message_consumer('Consumer A', env, pipe))

print('\nOne-to-one pipe communication\n')
env.run(until=SIM_TIME)

# For one-to many use BroadcastPipe
# (Note: could also be used for one-to-one,many-to-one or many-to-many)
env = simpy.Environment()
bc_pipe = BroadcastPipe(env)

env.process(message_generator('Generator A', env, bc_pipe))
env.process(message_consumer('Consumer A', env, bc_pipe.get_output_conn()))
env.process(message_consumer('Consumer B', env, bc_pipe.get_output_conn()))

print('\nOne-to-many pipe communication\n')
env.run(until=SIM_TIME)

Process communication

One-to-one pipe communication

at time 6: Consumer A received message: Generator A says hello at 6.
at time 12: Consumer A received message: Generator A says hello at 12.
at time 19: Consumer A received message: Generator A says hello at 19.
at time 26: Consumer A received message: Generator A says hello at 26.
at time 36: Consumer A received message: Generator A says hello at 36.
at time 46: Consumer A received message: Generator A says hello at 46.
at time 52: Consumer A received message: Generator A says hello at 52.
at time 58: Consumer A received message: Generator A says hello at 58.
LATE Getting Message: at time 66: Consumer A received message: Generator A says hello at 65
at time 75: Consumer A received message: Generator A says hello at 75.
at time 85: Consumer A received message: Generator A says hello at 85.
at time 95: Consumer A received message: Generator A says hello at 95.

One-to-many pipe communication

at time 10: Consumer A received message: G


**Salida.** En la comunicación uno a uno el consumidor recibe casi todos los mensajes en el mismo instante en
que se envían. La excepción es el mensaje de $t = 65$, que se leyó en $t = 66$ (marcado *LATE*): el consumidor
estaba ocupado y el mensaje esperó en el `Store`, que funciona como búfer. En la comunicación uno a muchos, la
clase `BroadcastPipe` tiene un `Store` por consumidor y copia cada mensaje en todos. Por eso A y B reciben lo
mismo, pero cada uno a su ritmo: B llegó tarde a los mensajes de 40 y 46, mientras que A los leyó a tiempo.


#### 4.7 Event Latency (latencia de eventos)

Modela un cable con un retardo de propagación de 10 unidades de tiempo. El emisor envía un mensaje cada 5
unidades, y el receptor los recibe 10 unidades después. El retardo vive en la clase `Cable` y no en el emisor
ni en el receptor, lo que permite cambiar el medio de transmisión sin tocar los procesos.

Elementos de SimPy: `Store`.

In [23]:
# Código del ejemplo oficial: https://simpy.readthedocs.io/en/latest/examples/latency.html
import simpy

SIM_DURATION = 100


class Cable:
    """This class represents the propagation through a cable."""

    def __init__(self, env, delay):
        self.env = env
        self.delay = delay
        self.store = simpy.Store(env)

    def latency(self, value):
        yield self.env.timeout(self.delay)
        self.store.put(value)

    def put(self, value):
        self.env.process(self.latency(value))

    def get(self):
        return self.store.get()


def sender(env, cable):
    """A process which randomly generates messages."""
    while True:
        # wait for next transmission
        yield env.timeout(5)
        cable.put(f'Sender sent this at {env.now}')


def receiver(env, cable):
    """A process which consumes messages."""
    while True:
        # Get event for message pipe
        msg = yield cable.get()
        print(f'Received this at {env.now} while {msg}')


# Setup and start the simulation
print('Event Latency')
env = simpy.Environment()

cable = Cable(env, 10)
env.process(sender(env, cable))
env.process(receiver(env, cable))

env.run(until=SIM_DURATION)

Event Latency
Received this at 15 while Sender sent this at 5
Received this at 20 while Sender sent this at 10
Received this at 25 while Sender sent this at 15
Received this at 30 while Sender sent this at 20
Received this at 35 while Sender sent this at 25
Received this at 40 while Sender sent this at 30
Received this at 45 while Sender sent this at 35
Received this at 50 while Sender sent this at 40
Received this at 55 while Sender sent this at 45
Received this at 60 while Sender sent this at 50
Received this at 65 while Sender sent this at 55
Received this at 70 while Sender sent this at 60
Received this at 75 while Sender sent this at 65
Received this at 80 while Sender sent this at 70
Received this at 85 while Sender sent this at 75
Received this at 90 while Sender sent this at 80
Received this at 95 while Sender sent this at 85



**Salida.** El emisor envía cada 5 unidades de tiempo y cada mensaje llega exactamente 10 unidades después (el
de $t = 5$ se recibe en 15, el de 10 en 20, y así). El retardo no está ni en el emisor ni en el receptor, sino en
`Cable.latency`, que lanza un proceso por mensaje que espera `delay` y luego lo deja en el `Store`. Como hay
un proceso por mensaje, varios mensajes pueden ir "por el cable" al mismo tiempo sin bloquearse entre sí.



---
### Resumen de los ejemplos

| Ejemplo | Qué modela | Elementos de SimPy |
|---|---|---|
| Bank Renege | Mostrador con clientes impacientes que abandonan | `Resource`, evento de condición `req \| timeout` |
| Carwash | Lavadero con 2 máquinas | `Resource`, esperar a otro proceso |
| Machine Shop | Máquinas que se dañan y un técnico con otras tareas | `interrupt()`, `PreemptiveResource` con prioridades |
| Movie Renege | Taquilla de cine con películas que se agotan | `Resource`, evento compartido, evento de condición |
| Gas Station Refueling | Surtidores con tanque común y carrotanque | `Resource`, `Container`, proceso de control |
| Process Communication | Envío de mensajes uno a uno y uno a muchos | `Store` como tubería y búfer |
| Event Latency | Cable con retardo de propagación | `Store`, un proceso por mensaje |

La diferencia con los numerales 5.1 a 5.3 está en el enfoque. Allá se programaba a mano la lista de sucesos
(`TSuc`), el reloj y la elección del suceso más próximo; es el enfoque de **programación de eventos**. En SimPy
se describe lo que hace cada entidad como un proceso (llegar, pedir un recurso, esperar, salir) y la librería se
encarga del reloj y de la cola de eventos; es el enfoque de **interacción de procesos**. Por eso modelos como el
taller de máquinas, con interrupciones y prioridades, caben en unas 100 líneas.
